# Part 6: The Trainer class

The `Trainer` class runs the loop for you: batching, the optimizer, the learning-rate schedule, evaluation and logging. It is the Hugging Face counterpart of Keras `fit`. This part runs the same fine-tuning as Part 5.

In [1]:
import warnings

from transformers.utils import logging

logging.set_verbosity_error()  # hide the long model-loading reports
warnings.filterwarnings("ignore")

from review_data import as_lists, load_reviews, small_sample

reviews = load_reviews()
val_texts, val_labels = as_lists(reviews["validation"])

texts, labels = small_sample(reviews["train"], 200, seed=42)

c:\Users\student\AppData\Local\anaconda3\envs\aigc-5005-module09-hf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from trainer_version import fine_tune_with_trainer

trainer, tokenizer = fine_tune_with_trainer(texts, labels, val_texts, val_labels)

Map: 100%|██████████| 1066/1066 [00:00<00:00, 9544.90 examples/s]


{'loss': '0.7036', 'grad_norm': '2.115', 'learning_rate': '3.857e-05', 'epoch': '1'}
{'eval_loss': '0.6725', 'eval_accuracy': '0.6013', 'eval_macro_f1': '0.5724', 'eval_runtime': '57.01', 'eval_samples_per_second': '18.7', 'eval_steps_per_second': '0.298', 'epoch': '1'}
{'loss': '0.6212', 'grad_norm': '4.074', 'learning_rate': '2e-05', 'epoch': '2'}
{'eval_loss': '0.6262', 'eval_accuracy': '0.6811', 'eval_macro_f1': '0.6735', 'eval_runtime': '42.62', 'eval_samples_per_second': '25.01', 'eval_steps_per_second': '0.399', 'epoch': '2'}
{'loss': '0.4807', 'grad_norm': '3.698', 'learning_rate': '1.429e-06', 'epoch': '3'}
{'eval_loss': '0.586', 'eval_accuracy': '0.6932', 'eval_macro_f1': '0.6911', 'eval_runtime': '50.61', 'eval_samples_per_second': '21.06', 'eval_steps_per_second': '0.336', 'epoch': '3'}
{'train_runtime': '273.1', 'train_samples_per_second': '2.197', 'train_steps_per_second': '0.143', 'train_loss': '0.6018', 'epoch': '3'}


In [3]:
for entry in trainer.state.log_history:
    if "eval_accuracy" in entry:
        print(round(entry["epoch"]), round(entry["eval_loss"], 4),
              round(entry["eval_accuracy"], 4))

1 0.6725 0.6013
2 0.6262 0.6811
3 0.586 0.6932


## How the two versions line up

| Your loop in Part 5 | The Trainer |
|---|---|
| `DataLoader` and `DataCollatorWithPadding` | `data_collator=DataCollatorWithPadding(tokenizer)` |
| `AdamW` and the linear schedule | `learning_rate`, `weight_decay`, `warmup_steps` in `TrainingArguments` |
| `loss.backward()`, `optimizer.step()` | inside `trainer.train()` |
| `predict` and `scores` | `compute_metrics` |
| the history list | `trainer.state.log_history` |

The numbers differ a little from Part 5 because the defaults differ, for example in how the warm-up is counted. That is normal. What matters is that you can say what each setting does.

## Try it

1. Change `eval_strategy` so that evaluation runs every 5 steps. What do you see?
2. Set `num_train_epochs` to 1. How far does the accuracy fall?
3. Find where the Trainer stores the learning rate it used at each logged step.